# Øvelser: Fra rå fil til document-term matrix

**Social Data Science 1: lektion 6**

I sidste uge hentede I data ned fra nettet. I dag arbejder vi med outputtet af sådan en scraping: 7.207 jobopslag fra fire portaler, indsamlet med fire forskellige scrapere i efteråret 2023.

Målet er ikke at analysere dem. Målet er at nå frem til en **datastruktur**, man kan
analysere og at kunne gøre rede for hvert eneste valg undervejs.

**Fil:** `jobapplications_fall-2023.csv`

**Sådan arbejder I:**

- Skriv hvert valg ned. De kan bruges i portfolien.
- Læs fejlbeskeder. De første fire opgaver består næsten kun af fejlbeskeder.


---

## Opgave 1: Få filen ind

**Trin 1: prøv det oplagte.** Kør cellen og læs fejlen.


In [1]:
import pandas as pd

opslag = pd.read_csv("/Users/jeppefl/Library/CloudStorage/OneDrive-AalborgUniversitet/01_work/01_undervisning/02_sds1/03_data/jobapplications/jobapplications_fall-2023.csv")


UnicodeDecodeError: 'utf-8' codec can't decode byte 0xbf in position 124: invalid start byte

**Trin 2: separatoren.**

Filen er lavet i et dansk regneark, hvor kommaet er decimaltegn. Derfor er
kolonnerne adskilt af noget andet.

Åbn filen i en teksteditor og kig på den første linje. Tilføj det rigtige `sep=`.


In [ ]:
# Din kode her



**Trin 3: overskriften.**

Nu skulle den være læst ind, men kolonnerne hedder noget forkert. Kig på de første to rækker.

*Hint:* `skiprows=`


In [ ]:
# Din kode her



**Trin 4: tegnsættet.**

Print et par titler. Ser `København` rigtig ud?

En tekstfil er bytes. Tegnsættet er aftalen om, hvilket tegn hver byte betyder, og
filen indeholder ingen oplysning om, hvilken aftale der gælder. Man gætter.

Prøv `encoding=` med `"utf-8"`, `"latin-1"`, `"cp1252"` og `"mac_roman"`.
Hvilken giver rigtige danske bogstaver?


In [ ]:
# Din kode her



### 1.5 Til diskussion

1. Hvad sker der, hvis I bruger den forkerte tegnsætning og ikke opdager det? Hvor i
   analysen ville det dukke op?


---

## Opgave 2: Metadata er også tekst

**Trin 1: hvad mangler?**

Kør `opslag.isna().sum()`. Kolonnen `company` mangler for de fleste opslag.

Undersøg om det er tilfældigt. Grupper efter `source` og se hvor mange der mangler
i hver kilde.


In [ ]:
# Din kode her



**Trin 2: find firmanavnet.**

Det er ikke væk. Print `various_meta1` for et opslag, hvor `company` mangler.

Hvad står der, og hvordan er felterne adskilt?


In [ ]:
# Din kode her



**Trin 3: hent det ud.**

Del feltet op med `.split("\n")`. Samme metode som i lektion 2, bare med
linjeskift i stedet for mellemrum.

Byg en ny kolonne `firma`, der indeholder den første linje.

*Hint:* `opslag["various_meta1"].str.split("\n").str[0]`


In [ ]:
# Din kode her



**Trin 4: fælden.**

Tæl, hvor mange linjer der er i hvert felt:

```python
opslag["various_meta1"].str.split("\n").str.len().value_counts()
```

De fleste har tre. Nogle har to. Kig på et par af dem med to linjer.

Hvad er der havnet i jeres `firma`-kolonne for netop de opslag? Og hvordan finder I
dem uden at læse alle 7.207 igennem?


In [ ]:
# Din kode her



**Trin 5: datoen.**

Tredje linje ser ud som `Indrykket 24-10-2023`. Hent datoen ud og gør den til en
rigtig dato.

Hvor mange lykkes det for? Og hvilke kilder har slet ingen dato?


In [ ]:
# Din kode her



---

## Opgave 3: Byg preprocessing-kæden

Nu til selve oprydningen. Efter hvert trin skal I notere **to** tal: antal tokens og
antal typer. 

**Trin 0: udgangspunktet.**

In [ ]:
tekster = opslag["job_description"].dropna().astype(str)

alle = [o for t in tekster for o in t.split()]

print("Tokens:", len(alle))
print("Typer: ", len(set(alle)))


**Trin 1: små bogstaver.**

Gæt først: hvilket af de to tal ændrer sig?


In [ ]:
# Din kode her



**Trin 2: tegnsætning.**

```python
import string
tegn = string.punctuation + "«»…–—"
```

Brug `.strip(tegn)` på hvert ord, og smid tomme strenge væk bagefter.

Hvorfor bliver `søger/finder` stående som ét ord?


In [ ]:
# Din kode her



**Trin 3: tal.**

Fjern alle ord, der indeholder et ciffer.

*Hint:* `any(c.isdigit() for c in o)`

Kig på ti af de ord, I lige har fjernet. Var de alle sammen støj?


In [ ]:
# Din kode her



**Trin 4: stopord.**


In [ ]:
stopord = set("""og i jeg det at en den til er som på de med han af for ikke
der var mig sig men et har om vi min havde ham hun nu over da fra du ud sin
dem os op man hans hvor eller hvad skal selv her alle vil blev kunne ind når
være dog jo dette dig deres end mit også under have dens hvis dine disse
hvem vores jer sådan andre nogle bliver blive kan""".split())

# Din kode her


---

## Opgave 7: spaCy

Denne del skal køres i UCloud eller lokalt.

```bash
pip install spacy
python -m spacy download da_core_news_sm
```

**Trin 1: kør pipelinen på én tekst.**

In [ ]:
import spacy

nlp = spacy.load("da_core_news_sm")
doc = nlp("Vi søger en erfaren socialrådgiver til vores team i Aarhus.")

for tok in doc:
    print(tok.text, tok.lemma_, tok.pos_, tok.is_stop)


**Trin 2: stemming mod lemmatisering.**

Kør pipelinen på ordene `søger`, `søgte`, `søgt`, `universitet`, `universel`.

Hvad bliver de til? Hvad ville en simpel regel — klip de sidste to bogstaver af —
have gjort ved de samme ord?


In [ ]:
# Din kode her



**Trin 3: ordklasser som filter.**

Kør pipelinen på 200 opslag og behold kun navneord og tillægsord.

```python
docs = nlp.pipe(tekster[:200], batch_size=50)
```

Sammenlign vokabularet med det, I fik af regelversionen på de samme 200 opslag.
Hvor stor er forskellen?


In [ ]:
# Din kode her



**Trin 4: find en fejl.**

Modellen er trænet på dansk nyhedstekst. Jobopslag er ikke nyhedstekst, og en del af
materialet er på engelsk.

Find mindst tre steder, hvor lemmatiseringen eller ordklassen er forkert. Hvilken
slags ord går det galt på?


In [ ]:
# Din kode her



### 7.5 Til diskussion

1. Reglerne kørte på sekunder. spaCy tager minutter. Hvornår er det tids-omkostningen værd?
2. I kan læse jeres egen stopordsliste. I kan ikke læse en model. Betyder det noget
   for, hvad I kan skrive i metodeafsnittet?


---

## Opgave 8: Til portfolien

Skriv en preprocessing-protokol for jeres eget materiale.

1. **Hvert valg, i rækkefølge.** Hvad gjorde I, og hvorfor netop det?

2. **Hvad konsekvensen ved hvert valg?** Ét konkret eksempel per trin på noget, der gik tabt.

3. **Tokens og typer** før og efter. Hvor stor blev reduktionen?

4. **Robusthed.** Prøvede I flere kombinationer? Holdt resultatet?

Protokollen skal være detaljeret nok til, at en anden kan gentage den præcist.
Det er hele pointen i Denny & Spirlings artikel.


In [ ]:
# Din kode her

